# 02_cuVS / 00 — Local multimodal embeddings with vLLM + EmbeddingGemma 2

Everything in this folder runs **locally on the GPU**: no API keys, no cloud storage, no paid calls. The embedding model is downloaded once from Hugging Face and runs on the T4.

| | |
|---|---|
| Model | [`google/embeddinggemma-2`](https://huggingface.co/google/embeddinggemma-2) (Apache 2.0, not gated) |
| Modalities | text, images, audio, video, and mixes of them, in **one** shared vector space |
| Output | 768 dims, normalized. MRL truncation to 512 / 256 / 128 |
| Size | 740M parameters (270M text, 170M vision, 300M audio) |
| Context | 8,192 tokens: image ≈ 280 tokens, video ≈ 140 tokens per frame at 1 fps, audio ≈ 25 tokens per second at 16 kHz mono |
| Inference | **vLLM** (`runner="pooling"`), with an automatic fallback to sentence-transformers |

This notebook:
1. Installs vLLM and checks the GPU
2. Loads the embedder (vLLM first, fallback if it can't start on this GPU)
3. Embeds sample text, images, audio and video, and checks cross-modal search
4. Tests interleaved inputs and MRL truncation
5. Measures throughput per modality, for vLLM and sentence-transformers

**Every notebook in this folder contains this same embedder cell**, so each one runs on its own.

### Two things specific to the T4
- **No bfloat16.** The model must not run in float16 (it gives NaN or silently wrong vectors). The T4 (compute capability 7.5) has no native bfloat16, so we run in **float32**. Newer GPUs use bfloat16 automatically.
- **Small shared memory (64 KB per block).** vLLM's attention kernel for this model has failed to start on GPUs with ≤ 99 KB ([vllm-project/vllm#60847](https://github.com/vllm-project/vllm/issues/60847)). If that happens here, the notebook prints the error and falls back to sentence-transformers. The vectors are the same, only slower.

## 1. Install

Installing vLLM takes a few minutes and replaces the runtime's PyTorch. **Restart the kernel after this cell**, then run the rest.

In [ ]:
import importlib.util, subprocess, sys

def has(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False

pkgs = []
if not has("vllm"):
    pkgs.append("vllm==0.32.0")
pkgs += ["sentence-transformers>=5.4", "librosa", "soundfile", "opencv-python-headless", "scikit-image"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs])
print("Installed. Restart the kernel now, then continue from section 2.")

### Hugging Face token (optional)

`google/embeddinggemma-2` is **not gated**, so no token is needed. A token only raises download rate limits. If you want one, put it in Colab Secrets as `HF_TOKEN`, or set the environment variable before loading the model:

```python
import os; os.environ["HF_TOKEN"] = "hf_..."   # never commit this
```

## 2. Environment

In [ ]:
import os, json, time, gc, platform
import numpy as np, torch

props = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
smem = getattr(props, "shared_memory_per_block_optin", None)

env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda (torch)": torch.version.cuda,
    "gpu": props.name,
    "gpu_mem_gb": round(props.total_memory / 1e9, 1),
    "compute_capability": f"{cc[0]}.{cc[1]}",
    "shared_mem_per_block_optin_kb": smem // 1024 if smem else None,
}
for mod in ["vllm", "transformers", "sentence_transformers"]:
    try:
        env[mod] = __import__(mod).__version__
    except Exception as e:
        env[mod] = f"not available ({type(e).__name__})"
env

## 3. The embedder

This is the cell every notebook in the folder repeats. Inputs are plain dicts:

| Item | Dict |
|---|---|
| search query | `{"text": "task: search result \| query: ..."}` |
| text document | `{"text": "title: ... \| text: ..."}` |
| image | `{"text": "<\|image\|>", "image": PIL.Image}` |
| audio | `{"text": "<\|audio\|>", "audio": "clip.wav"}` |
| video | `{"text": "<\|video\|>", "video": "clip.mp4"}` |
| mixed | `{"text": "Red shoes <\|image\|> on wet rock <\|video\|>", "image": img, "video": "demo.mp4"}` |

Media items get no task prefix (as the model card recommends). `<|image|>`, `<|audio|>` and `<|video|>` mark where each item sits in the sequence.

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BACKEND  = "auto"          # "auto" (vLLM, else sentence-transformers) | "vllm" | "sentence-transformers"
MAX_LEN  = 8192
VLLM_GPU_MEM = 0.45        # fraction of GPU memory vLLM may reserve (leave room for cuVS)
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0) >= (8, 0) else torch.float32   # never float16

def query_item(q):            return {"text": f"task: search result | query: {q}"}
def doc_item(text, title="none"): return {"text": f"title: {title} | text: {text}"}
def image_item(img):          return {"text": "<|image|>", "image": img}
def audio_item(path):         return {"text": "<|audio|>", "audio": path}
def video_item(path):         return {"text": "<|video|>", "video": path}

def _normalize(x, dim=None):
    x = np.asarray(x, dtype=np.float32)
    if dim is not None:
        x = x[:, :dim]
    return x / np.linalg.norm(x, axis=1, keepdims=True)

class Embedder:
    """EmbeddingGemma 2 on vLLM, falling back to sentence-transformers."""

    def __init__(self, backend=BACKEND):
        self.backend, self.vllm_error = None, None
        if backend in ("auto", "vllm"):
            try:
                from vllm import LLM
                self.llm = LLM(model=MODEL_ID, runner="pooling",
                               dtype="bfloat16" if DTYPE == torch.bfloat16 else "float32",
                               max_model_len=MAX_LEN, gpu_memory_utilization=VLLM_GPU_MEM,
                               enforce_eager=True,
                               limit_mm_per_prompt={"image": 4, "audio": 1, "video": 1})
                self.backend = "vllm"
            except Exception as e:
                self.vllm_error = f"{type(e).__name__}: {e}"
                if backend == "vllm":
                    raise
                print("vLLM could not start on this GPU, using sentence-transformers instead.")
                print("  reason:", self.vllm_error[:800])
                gc.collect(); torch.cuda.empty_cache()
        if self.backend is None:
            from sentence_transformers import SentenceTransformer
            self.st = SentenceTransformer(MODEL_ID, device="cuda", model_kwargs={"torch_dtype": DTYPE})
            self.backend = "sentence-transformers"
        print(f"embedding backend: {self.backend}  ({str(DTYPE).replace('torch.', '')})")

    # --- vLLM -----------------------------------------------------------------
    @staticmethod
    def _to_vllm(item):
        mm = {}
        if "image" in item:
            mm["image"] = item["image"]
        if "audio" in item:
            import librosa
            wav, sr = librosa.load(item["audio"], sr=16_000, mono=True)
            mm["audio"] = (wav, sr)
        if "video" in item:
            from transformers.video_utils import load_video
            mm["video"] = load_video(item["video"], backend="opencv")
        return {"prompt": item["text"], "multi_modal_data": mm} if mm else item["text"]

    # --- public ---------------------------------------------------------------
    def embed(self, items, dim=768, batch_size=16):
        """items: list of dicts (see table above) -> float32 array (len(items), dim), L2-normalized."""
        if self.backend == "vllm":
            out = []
            for i in range(0, len(items), batch_size * 8):          # bound host memory for decoded media
                prompts = [self._to_vllm(it) for it in items[i:i + batch_size * 8]]
                out += [o.outputs.embedding for o in self.llm.embed(prompts, use_tqdm=False)]
            return _normalize(out, dim)
        st_items = [it["text"] if set(it) == {"text"} else it for it in items]
        vecs = self.st.encode(st_items, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)
        return _normalize(vecs, dim)

emb = Embedder()

## 4. Sample media

Small local samples for a first check (the real datasets come in notebook 01):
- **Images:** four photos bundled with scikit-image
- **Audio:** four clips from librosa's example set (whale song, trumpet, speech, drums and bass), resampled to 16 kHz mono
- **Video:** two short clips generated here (a bouncing red ball, and a slow zoom into the cat photo)

In [ ]:
import cv2, soundfile as sf, librosa
from PIL import Image
from skimage import data as skdata

ASSETS = "assets"
os.makedirs(ASSETS, exist_ok=True)

images = {
    "an astronaut": Image.fromarray(skdata.astronaut()),
    "a cat":        Image.fromarray(skdata.chelsea()),
    "a cup of coffee": Image.fromarray(skdata.coffee()),
    "a rocket":     Image.fromarray(skdata.rocket()),
}

audio_src = {"humpback": "whale song", "trumpet": "trumpet", "libri1": "speech", "choice": "drums and bass"}
audios = {}
for key, label in audio_src.items():
    path = os.path.join(ASSETS, f"{key}.wav")
    if not os.path.exists(path):
        wav, sr = librosa.load(librosa.ex(key), sr=16_000, mono=True, duration=15)
        sf.write(path, wav, sr)
    audios[label] = path

def write_video(path, frames, fps=10):
    h, w = frames[0].shape[:2]
    vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames:
        vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    vw.release()

ball_path, zoom_path = os.path.join(ASSETS, "red_ball.mp4"), os.path.join(ASSETS, "cat_zoom.mp4")
if not os.path.exists(ball_path):
    frames = []
    for t in range(80):                                   # 8 s at 10 fps
        f = np.full((240, 320, 3), 235, np.uint8)
        y = int(200 - 160 * abs(np.sin(t / 8)))
        cv2.circle(f, (40 + 3 * t, y), 18, (220, 30, 30), -1)
        cv2.line(f, (0, 220), (320, 220), (90, 90, 90), 3)
        frames.append(f)
    write_video(ball_path, frames)
if not os.path.exists(zoom_path):
    cat = skdata.chelsea()
    H, W = cat.shape[:2]
    frames = []
    for t in range(80):
        s = 1 - 0.5 * t / 80
        h, w = int(H * s), int(W * s)
        crop = cat[(H - h) // 2:(H + h) // 2, (W - w) // 2:(W + w) // 2]
        frames.append(cv2.resize(crop, (320, 240)))
    write_video(zoom_path, frames)
videos = {"a red ball bouncing": ball_path, "a cat (zooming in)": zoom_path}

print(f"{len(images)} images | {len(audios)} audio clips | {len(videos)} videos")

## 5. Cross-modal search

One pool of 10 items (4 images, 4 sounds, 2 videos), searched with **text** queries. Each query should land on its own item, across modalities.

In [ ]:
items  = [image_item(v) for v in images.values()] + [audio_item(v) for v in audios.values()] + [video_item(v) for v in videos.values()]
labels = [f"[image] {k}" for k in images] + [f"[audio] {k}" for k in audios] + [f"[video] {k}" for k in videos]

t0 = time.perf_counter()
M = emb.embed(items)
print(f"embedded {len(items)} media items in {time.perf_counter() - t0:.2f} s -> {M.shape}")

queries = ["an astronaut in a space suit", "a cat", "a cup of coffee", "a rocket launch",
           "a whale singing underwater", "a trumpet playing", "a person reading a book aloud",
           "drums and bass music", "a red ball bouncing on the floor", "a close-up of a cat"]
Qv = emb.embed([query_item(q) for q in queries])
S = Qv @ M.T

hits = (S.argmax(1) == np.arange(len(queries))).sum()
print(f"top-1 correct: {hits} / {len(queries)}")
for q, row in zip(queries, S):
    print(f"  {q:<34} -> {labels[row.argmax()]:<32} ({row.max():.3f})")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5.5))
im = ax.imshow(S, cmap="viridis")
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(queries)), queries, fontsize=8)
for i, j in enumerate(S.argmax(1)):
    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="white", lw=2))
fig.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title(f"Text queries vs images, audio and video ({emb.backend})")
plt.tight_layout(); plt.show()

## 6. Text search, interleaved inputs and MRL

In [ ]:
docs = [
    ("Aurora", "The northern lights are caused by charged particles from the sun colliding with gases in the atmosphere."),
    ("Turing (microarchitecture)", "Turing is an NVIDIA GPU architecture used in the Tesla T4 and the GeForce RTX 20 series."),
    ("Mango", "The mango is an edible stone fruit produced by the tropical tree Mangifera indica."),
]
D = emb.embed([doc_item(t, title) for title, t in docs])
Qt = emb.embed([query_item("what causes the aurora borealis"), query_item("which gpu architecture does the T4 use"),
                query_item("tropical fruit")])
print("query x doc similarity:\n", (Qt @ D.T).round(3))

In [ ]:
# Interleaved: one embedding for text + image together, compared with text queries
mixed = emb.embed([{"text": "Space mission photo: <|image|> crew member ready for launch", "image": images["an astronaut"]},
                   {"text": "Breakfast table: <|image|> freshly brewed", "image": images["a cup of coffee"]}])
mq = emb.embed([query_item("astronaut preparing for a space launch"), query_item("morning coffee")])
print("query x mixed item:\n", (mq @ mixed.T).round(3))

In [ ]:
# MRL: shorter vectors (re-normalized) should rank the same items first
full = (Qv @ M.T).argmax(1)
for d in [512, 256, 128]:
    s = emb.embed([query_item(q) for q in queries], dim=d) @ _normalize(M, d).T
    print(f"{d:>4} dims: top-1 agrees with 768 dims on {(s.argmax(1) == full).sum()} / {len(queries)} queries "
          f"| {d * 4} bytes per vector")

## 7. Throughput

How many items per second the embedder handles for each modality on this GPU. Text uses ~500-token passages, like the corpus in notebook 01. If vLLM is running, sentence-transformers is measured too, for comparison.

In [ ]:
words = ("The quick brown fox jumps over the lazy dog while the GPU embeds long passages of text " * 30).split()
passage = " ".join(words[:375])                                   # ~500 tokens
bench_sets = {
    "text (500 tok)": [doc_item(passage + f" {i}", "Benchmark") for i in range(256)],
    "image":          [image_item(list(images.values())[i % 4]) for i in range(64)],
    "audio (15 s)":   [audio_item(list(audios.values())[i % 4]) for i in range(32)],
    "video (8 s)":    [video_item(list(videos.values())[i % 2]) for i in range(16)],
}

def throughput(e, its):
    e.embed(its[:4]); torch.cuda.synchronize()             # warmup
    t0 = time.perf_counter(); e.embed(its); torch.cuda.synchronize()
    return len(its) / (time.perf_counter() - t0)

tput = {emb.backend: {k: throughput(emb, v) for k, v in bench_sets.items()}}

if emb.backend == "vllm":
    st_emb = Embedder("sentence-transformers")
    tput["sentence-transformers"] = {k: throughput(st_emb, v) for k, v in bench_sets.items()}
    # same vectors from both backends?
    a, b = emb.embed(items[:4]), st_emb.embed(items[:4])
    print("vLLM vs sentence-transformers cosine (images):", np.diag(a @ b.T).round(4))
    del st_emb; gc.collect(); torch.cuda.empty_cache()

import pandas as pd
tp = pd.DataFrame(tput).round(1)
tp.index.name = "items / second"
tp

In [ ]:
ax = tp.plot.barh(figsize=(8, 3.2), logx=True)
ax.set_xlabel("items per second (log, higher is better)")
ax.set_title(f"EmbeddingGemma 2 throughput on {props.name} ({str(DTYPE).replace('torch.', '')})")
plt.tight_layout(); plt.show()

os.makedirs("results", exist_ok=True)
with open("results/00_embedder_check.json", "w") as f:
    json.dump({"env": env, "backend": emb.backend, "vllm_error": emb.vllm_error,
               "dtype": str(DTYPE), "cross_modal_top1": int(hits), "throughput": tput}, f, indent=2)
print("saved results/00_embedder_check.json")

## Summary

- EmbeddingGemma 2 runs **locally**: no API key, nothing leaves the runtime.
- One model puts **text, images, audio and video** in the same 768-dim space, so a text query can find a sound or a video.
- On the T4 it runs in **float32** (no bfloat16, and float16 is unsafe for this model).
- The `Embedder` cell above (vLLM with automatic fallback) is copied into every notebook in this folder.
- Throughput per modality tells us how big a corpus we can embed in reasonable time in notebook 01.

**Next:** `01_build_multimodal_corpus.ipynb` — real datasets for every modality, embedded and saved locally.